In [24]:
import pandas as pd
import numpy as np
from pathlib import Path
import pantab


In [25]:
# Input file paths

billing_raw = pd.read_excel(r'C:\Users\pirat\Dropbox\Consulting Inc\Advanced Dermatology\1. Payor Data & Contracts'
    r'\Arizona\Cigna Analysis\PowerBI'
    r'\Cigna AZ - DOS 2024 & 2025 - as of 4.6.26 - PowerBI.xlsx')

service_map = pd.read_excel(r'C:\Users\pirat\Dropbox\Consulting Inc\Josh Perkins Data\CMS Data\Service Category Mapping.xlsx')

billing_clean = billing_raw.copy()

In [26]:
# Commonly changed variables

#output file path
output_dir = Path(r"C:\Users\pirat\Dropbox\Consulting Inc\Advanced Dermatology\1. Payor Data & Contracts"
                  r"\Arizona\Cigna Analysis\Tableau")
base = output_dir / "Arizona_Cigna_Billing_Cleaned"

exports = {
    "xlsx":  base.with_suffix(".xlsx"),
    "hyper": base.with_suffix(".hyper"),
}



# Contract Designation: Specify New contract date range
contract_start_date = '2025-01-01'
contract_end_date = '2025-12-31'

# "Old Contract" period will be exactly one year prior to "New Contract" period dynamically
old_contract_start_date = pd.to_datetime(contract_start_date) - pd.DateOffset(years=1)
old_contract_end_date = pd.to_datetime(contract_end_date) - pd.DateOffset(years=1)

# Localities for matching to CMS MAC codes if needed.
# print unique values in Location Name
print(billing_clean['Location Name'].unique())

locality_map = {
"SC-Greenville-450": "Green/Spartan - RHP",
"Spartanburg 265": "Green/Spartan - RHP",
"SC-Ropers Corner-Greenville-455": "Green/Spartan - RHP",
"SC-West Columbia-Sunset-443": "Ft Mill / Columbia - Direct",
"SC-Fort Mill-446": "Ft Mill / Columbia - Direct",
"SC-Arden Ft Mill-461": "Ft Mill / Columbia - Direct",
"SC-North Columbia-508": "Ft Mill / Columbia - Direct"
}

####
# Michigan
# 'DANM- Alpena EMR 396' : 'Lower North',
# 'GSI- Shelby Township 276' : 'Detroit',
# 'GSI- Warren 277' : 'Detroit',
# 'MI-Bloomfield-Hills-506' : 'Detroit',
# 'Livonia MI 322' : 'Detroit',
# 'GSI- Wyandotte 278' : 'Detroit',
# 'GSI- Franklin 275' : 'Detroit',
# 'Kalamazoo MI 359' : 'South Central',
# 'MMD- Lansing MI 337' : 'South Central',
# 'DANM- Cheboygan EMR 8003' : 'Lower North',
# 'DANM- Gaylord EMR 8005' : 'Lower North',
# 'DANM- Petoskey EMR 319' : 'Lower North',
# 'MI-Kalkaska-W Mile-444' : 'Lower North',
# 'Traverse City MI 336' : 'Lower North',
# 'MI-Silver Pines-Macomb-503' : 'Detroit'

# Pennsylvania
# "PID- Ft Washington-OCD 360"        : "Philadelphia",
# "Beaver-PA-Stiegel-387"             : "Pittsburgh",
# "MBD- Roxborough-304"               : "Philadelphia",
# "Pittsburgh-PA-Martone-386"         : "Pittsburgh",
# "PID- Flourtown 362"                : "Philadelphia",
# "MBD- Hazelton-301"                 : "Scranton",
# "PA-King of Prussia-302"            : "Philadelphia",
# "PID- North Wales-Montgomery 363"   : "Philadelphia",
# "Goldman 335"                       : "Philadelphia",
# "DA-Pittsburgh - Jefferson 366"     : "Pittsburgh",
# "DA-White Oak 367"                  : "Pittsburgh",
# "PID- Flourtown 362B"               : "Philadelphia"

# Maryland
# 'GWD- Annapolis EMR 349' : 'Baltimore',
# 'MD-Towson-449' : 'Baltimore',
# 'MD-Abingdon-447' : 'Baltimore',
# 'MD-Cockeysville-448' : 'Baltimore',
# 'MD-Pikesville-460' : 'Baltimore',
# 'La Plata MD- 0436' : 'DC-Arlington',
# 'GWD- Olney EMR 347' : 'DC-Arlington',
# 'GWD- Rockville EMR 346' : 'DC-Arlington',
# 'Fort Washington MD-0435' : 'DC-Arlington',
# 'National Harbor-MD-420' : 'DC-Arlington',
# 'MD-Maragh-Rockville-458' : 'DC-Arlington'

# South Carolina
# "SC-West Columbia-Sunset-443" : "Mid-Columbia",
# "Spartanburg 265" : "Upstate",
# "SC-Greenville-450" : "Upstate",
# "SC-Arden Ft Mill-461" : "Upstate",
# "SC-Fort Mill-446" : "Upstate",
# "SC-Ropers Corner-Greenville-455" : "Upstate",
# "SC-North Columbia-508" : "Mid-Columbia"

# "SC-Greenville-450": "Green/Spartan - RHP",
# "Spartanburg 265": "Green/Spartan - RHP",
# "SC-Ropers Corner-Greenville-455": "Green/Spartan - RHP",
# "SC-West Columbia-Sunset-443": "Ft Mill / Columbia - Direct",
# "SC-Fort Mill-446": "Ft Mill / Columbia - Direct",
# "SC-Arden Ft Mill-461": "Ft Mill / Columbia - Direct",
# "SC-North Columbia-508": "Ft Mill / Columbia - Direct"

# Florida
# 'Boca Raton 120': '03',
# 'Bokeelia 374': '03',
# 'Cape Coral 375': '03',
# 'FL-Boynton-Seacrest-442': '03',
# 'FL-FSC-Cape Coral-451': '03',
# 'FL-FSC-Fort Myers-454': '03',
# 'FL-FSC-Fort Myers-456': '03',
# 'FL-FSC-Lehigh Acres-453': '03',
# 'Fort Myers 178': '03',
# 'Pathology Florida': '03',
# 'Pembroke Pines (was Miramar) 262': '03',
# 'Port St. Lucie - 393': '03',
# 'Sebastian 161': '03',
# 'Vero Terrace 429': '03',
# 'Weston 117': '03',
# 'Islamorada 195': '04',
# 'Baldwin Park 268': '99',
# 'Brandon 111': '99',
# 'Brownwood 378': '99',
# 'CD- Port Charlotte 354': '99',
# 'CD- Punta Gorda 355': '99',
# 'CDSS- Deleon 251': '99',
# 'CDSS- New Port Richey 250': '99',
# 'CDSS- Spring Hill 255': '99',
# 'CDSS- Wesley Chapel 254': '99',
# 'CDSS-University 0253': '99',
# 'Clearwater Countryside 424': '99',
# 'Clermont 133': '99',
# 'Deland -127': '99',
# 'Deland-Plymouth 326': '99',
# 'Deltona - Debary 103': '99',
# 'FL-Altamonte-Fern Park-113': '99',
# 'FL-Brownwood-507': '99',
# 'FL-FSC-Punta Gorda-452': '99',
# 'FL-Villages-Colony-439': '99',
# 'Heathrow 101': '99',
# 'Hunters Creek 118': '99',
# 'Jacksonville - Chimney Lakes 176': '99',
# 'Jacksonville - Philips Hwy-Bayard -175': '99',
# 'Jacksonville - San Marco- 428': '99',
# 'Julington Creek 261': '99',
# 'Lake Faith 152': '99',
# 'Lake Nona 373': '99',
# 'Lake Sumter Landing 170': '99',
# 'Lakeland -199': '99',
# 'Largo 119': '99',
# 'Lee Road 194': '99',
# 'Lee Road 194-B': '99',
# 'Maitland - For Misc Payments': '99',
# 'Metrowest 104': '99',
# 'New Smyrna 377': '99',
# 'NFDA - Orange Park Kingsley Ave 272': '99',
# 'NFDA - St Augustine South Park Blvd 274': '99',
# 'Ocala 174': '99',
# 'Orlando - S Orange Ave 105': '99',
# 'Ormond Beach 136': '99',
# 'Oviedo 102': '99',
# 'Palm Coast 171': '99',
# 'Palm Coast- Park Dr 260': '99',
# 'Palm Harbor- Woodlands-197': '99',
# 'Pathology Winter Park Saeed': '99',
# 'Plant City EMR 187': '99',
# 'Ponte Vedra FL-379': '99',
# 'Port Orange 131': '99',
# 'Riverview 430': '99',
# 'Spring Hill Drive-FL-385': '99',
# 'St Cloud 172': '99',
# 'St Pete -  Carillon Pkwy-0320': '99',
# 'St Petersburg 124': '99',
# 'Tavares 163': '99',
# 'The Villages 135': '99',
# 'Venice -  EMA 338': '99',
# 'Waterford Lakes 106': '99',
# 'Windermere 433': '99',
# 'Zephyrhills 185': '99',



['AAD- North Scottsdale 323' 'SD-Scottsdale Oldtown -AZ-408'
 'AAD- Central Phoenix 324' 'AAD- Central Phoenix-MOHS-324A'
 'AAD- Gilbert 325' 'AZ-Avondale-477' nan]


In [27]:
# Initial cleanup

# if "Payer" is null delete row
billing_clean = billing_clean.dropna(subset=['Payer'])

# replace col header spaces with underscores
billing_clean.columns = billing_clean.columns.str.replace(' ', '_')
service_map.columns = service_map.columns.str.replace(' ', '_')

# if "Provider SubGrp 1" is anything other than "Physicians" or "Extenders", change to "Physicians"
billing_clean.loc[~billing_clean['Provider_SubGrp_1'].isin(['Physicians', 'Extenders']), 'Provider_SubGrp_1'] = 'Physicians'

# CPT Code columns based on Service Item
billing_clean['CPT_Code_Full'] = billing_clean['Service_Item'].str.split(' - ').str[0]
billing_clean['Service_Description'] = billing_clean['Service_Item'].str.split(' - ').str[1]
billing_clean['CPT_Code_Core'] = billing_clean['CPT_Code_Full'].str.split('-').str[0]

# join service map "Service Category" to billing_clean on "CPT_Code_Core" and "CPT Code"
billing_clean = billing_clean.merge(service_map[['Code_Full', 'Category_Name']], 
                                    left_on='CPT_Code_Full', right_on='Code_Full', how='left')

# rename "Category_Name" to "Service_Category"
billing_clean = billing_clean.rename(columns={'Category_Name': 'Service_Category'})

In [28]:
# Date cleanup

# if year is missing, delete row
billing_clean = billing_clean.dropna(subset=['Year'])

# change year from float to int
billing_clean['Year'] = billing_clean['Year'].astype(int)

# Create date from 'Year' and 'Date_-_Month' columns with day as 1
billing_clean['Date'] = pd.to_datetime(billing_clean['Year'].astype(str) + '-' + billing_clean['Date_-_Month'].astype(str) + '-01')

In [29]:
# Calculated Columns

# new col "Avg Alwd" == "Act_Alwd" / "Count"
billing_clean['Avg_Alwd'] = billing_clean['Act_Alwd'] / billing_clean['Count']

# new col "Total_Paid" == "Act_TP_Paid*" + "Pat_Paid"
billing_clean['Total_Paid'] = (
    pd.to_numeric(billing_clean['Act_TP_Paid*'], errors='coerce')
      .add(pd.to_numeric(billing_clean['Pat_Paid'], errors='coerce'), fill_value=0)
      .round(5)
)*-1

# new col "Avg_Paid" == "Total_Paid" / "Count"
billing_clean['Avg_Paid'] = billing_clean['Total_Paid'] / billing_clean['Count']

In [30]:
# Payor Mapping to LOB

# First make a series based on the raw insurance name
lob = billing_clean["Payer_Sub_Group_2"].fillna("").str.lower()

# build a guess for every row
payor_guess = np.select(
    [
        lob.str.contains("medicare") | lob.str.contains("mcr"),
        lob.str.contains("medicaid") | lob.str.contains("mcd"),
        lob.str.contains("tricare") | lob.str.contains("exchange"),
    ],
    ["MCR", "MCD", "Exchange"],
    default="Commercial",
)

# map the guesses back to the dataframe
billing_clean["Payor_LOB"] = payor_guess

# apply any overrides based on contract name
overrides = {
    # "AETNA Commercial FL Loc 99": "TEST"
}

billing_clean["Payor_LOB"] = (
    billing_clean["Contract_Name"].map(overrides)
    .fillna(billing_clean["Payor_LOB"])
)


In [31]:
# Add cols for commonly changed variables

# Identify if rows are in the new contract period (New, Old, or Neither)
billing_clean['Contract_Period'] = np.where(
    (billing_clean['Date'] >= pd.to_datetime(contract_start_date)) & (billing_clean['Date'] <= pd.to_datetime(contract_end_date)),
    'New Contract',
    np.where(
        (billing_clean['Date'] >= old_contract_start_date) & (billing_clean['Date'] <= old_contract_end_date),
        'Old Contract',
        'Neither'
    )
)

# Add locality
billing_clean['Locality'] = billing_clean['Location_Name'].map(locality_map).fillna("Unknown")


In [32]:
#re-order columns to have the new ones closer to the front
cols = billing_clean.columns.tolist()
new_order = ['CPT_Code_Full', 'CPT_Code_Core', 'Service_Description', 'Service_Category',
      'Payor_LOB', 'Date', 'Avg_Alwd', 'Total_Paid', 'Avg_Paid',  'Contract_Period', 'Locality'     
]
cols = new_order + [c for c in cols if c not in new_order]
billing_clean = billing_clean[cols]

In [33]:
# Make a Hyper-safe copy (mainly: booleans)
billing_hyper = billing_clean.copy()

bool_cols = billing_hyper.select_dtypes(include=["bool", "boolean"]).columns
if len(bool_cols):
    # keep nulls while avoiding Arrow bool type (works well with Hyper)
    billing_hyper[bool_cols] = billing_hyper[bool_cols].astype("Int8")  # or "string"

writers = {
    "xlsx":  lambda df, p: df.to_excel(p, index=False),
    "hyper": lambda df, p: pantab.frame_to_hyper(df, str(p), table="Extract"),
}

for fmt, path in exports.items():
    df_to_write = billing_hyper if fmt == "hyper" else billing_clean
    writers[fmt](df_to_write, path)
